# 10 联合 A*：联合状态空间


本课按百科条目写。多车问题第一次正面出现：把 k 台车的位置绑成一个状态再搜。调度台在最后。

---

## 1. 定义

k 台智能体的 **联合状态** 是位置元组 \(S=(p_1,\ldots,p_k)\)。**联合 A\*** 在这张隐式图上做 A\*，通常最小化 SOC（各车到达时间和）或 makespan（最慢车）。

后继：每台车选「走一个邻居」或「停」，做笛卡尔积，再删掉

- **顶点冲突**：两车同一格
- **边冲突 / 换位**：A 走到 B 刚才的格子，同时 B 走到 A 刚才的格子

- 输入：图、各车起终点、扩展上限 `cap`。
- 输出：每车一条同步的点列（同一下标 = 同一拍），或因 cap 失败。

单位代价、有限图上 **完备且最优**。只适合 k 很小。

---

## 2. 和单车 A* / 后面各课

| | 单车 A\* | 联合 A\* | HCA\* / CBS |
|--|--|--|--|
| 状态 | 一个点 | k 个点的元组 | 单车 + 约束或优先级 |
| 最优 | 是 | 是（SOC/makespan 看你怎么写 g） | HCA 否；CBS 是 |
| 分支 | ≤4 | \(\approx 5^k\) | 单车级 |

联合 A\* 是正确性的「金标准」：两车小图上，后面任何最优算法的 SOC 都不应更差。它不是工程方案。

---

## 3. 不变量

- g(S) = 从起点联合状态走到 S 的累计移动次数和（本课等待不计 SOC，与库 `soc` 去掉终点停留一致）。
- 弹出目标联合状态时 g 最优。
- 每个后继必须 `_config_ok`：位置两两不同，且无对向换位。

**复杂度：** 每步分支约 \(5^k\)（四连通+等待）。深度 d 时树 \(5^{kd}\)。3 车在 5×5 上空跑，cap 几百就会触顶。

---

## 4. 伪代码

```
start ← (s1,...,sk); goal ← (g1,...,gk)
g[start]←0; push (h_sum, start)
while heap 且 expanded < cap:
    S ← pop
    if S = goal: 回溯，拆成每车路径
    for 每车动作组合 C:          # product
        if 顶点冲突或换位: skip
        ng ← g[S] + (非等待步数)
        松弛 C，h = Σ manhattan(pi, gi)
```

---

## 5. 手算（为什么必须能等待）

两车在两格走廊对换：

```
A: a→  ←b :B     目标互换
```

若不许等待，同时迈步只能换位 → 边冲突，被删。允许一台等、另一台……仍然过不去，因为没有第三格。这张图联合 A\* 应失败（无解），不是 bug。

换到 3×2 空网格，a 从 (0,0) 到 (2,0)，b 从 (2,1) 到 (0,1)，两人各走对边，不必相遇。联合状态起点 `((0,0),(2,1))`，目标 `((2,0),(0,1))`。


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "lib").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))


## 6. 实现：两车联合 A*（从零写）


In [ ]:
import heapq
import itertools
from lib.maps import grid_open
from lib.pretty import draw_grid
from lib.agents import soc, is_conflict_free


def manhattan(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])


def joint_astar_tiny(graph, starts, goals, cap=5000, verbose=True):
    # starts/goals: 长度为 2 的元组，顺序固定 (a, b)
    start_s, goal_s = starts, goals
    heap = [(0, 0, start_s)]
    g = {start_s: 0}
    came = {start_s: None}
    expanded = 0
    while heap and expanded < cap:
        _f, cost, state = heapq.heappop(heap)
        expanded += 1
        if verbose and expanded <= 12:
            print(f"扩展#{expanded} {state} g={cost}")
        if state == goal_s:
            chain = []
            cur = state
            while cur is not None:
                chain.append(cur)
                cur = came[cur]
            chain.reverse()
            paths = {"a": [st[0] for st in chain], "b": [st[1] for st in chain]}
            return paths, soc(paths), expanded
        moves = []
        for i in range(2):
            opts = [state[i]] + [n for n, _ in graph.neighbors(state[i])]
            moves.append(opts)
        for combo in itertools.product(*moves):
            if combo[0] == combo[1]:
                continue
            # 对向换位
            if combo[0] == state[1] and combo[1] == state[0] and combo[0] != state[0]:
                continue
            ng = cost + sum(0 if combo[i] == state[i] else 1 for i in range(2))
            if ng < g.get(combo, float("inf")):
                g[combo] = ng
                came[combo] = state
                h = manhattan(combo[0], goal_s[0]) + manhattan(combo[1], goal_s[1])
                heapq.heappush(heap, (ng + h, ng, combo))
    return None, float("inf"), expanded


g = grid_open(3, 2)
print(draw_grid(3, 2, start=(0, 0), goal=(2, 0)))
print("b 从 (2,1) 到 (0,1)，与 a 错开一行")
paths, cost, n_exp = joint_astar_tiny(g, ((0, 0), (2, 1)), ((2, 0), (0, 1)))
print("路径 a", paths["a"])
print("路径 b", paths["b"])
print("SOC", cost, "扩展", n_exp, "无冲突", is_conflict_free(paths))


## 7. 逐行

| 行 | 作用 |
|----|------|
| `state` 二元组 | 联合位置，可哈希，能进 dict |
| `opts = [自己] + 邻居` | 等待 |
| `itertools.product` | 两车动作笛卡尔积 |
| `combo[0]==combo[1]` | 顶点冲突 |
| 换位三项 | 对向穿过 |
| `ng` 非等待步数和 | 教学 SOC |
| `h` 两车曼哈顿和 | 可采纳（忽略相互阻挡，放宽问题） |

---

## 8. 库对照：2 车能解，3 车 cap 就爆


In [ ]:
from evals.scenarios import two_agent_swap
from lib.algos.prioritized import joint_astar

g2, starts, goals = two_agent_swap()
r = joint_astar(g2, starts, goals, cap=20000)
print("2车 found", r["found"], "SOC", r["soc"], "扩展", r["expanded"])

g3 = grid_open(5, 5)
r3 = joint_astar(
    g3,
    {"a": (0, 0), "b": (4, 0), "c": (0, 4)},
    {"a": (4, 4), "b": (0, 4), "c": (4, 0)},
    cap=400,
)
print("3车 cap400 扩展", r3["expanded"], "capped", r3.get("capped"), "found", r3["found"])


3 车 cap=400 几乎必然 `capped=True`。这就是后面要拆开搜的原因：HCA 按优先级、CBS 按冲突分支，状态不再是 \(5^k\)。

## 常见 bug

1. 漏边冲突：对向穿过，顶点在下一拍已经错开。
2. 漏等待：死锁无法「谁先停」。
3. closed 只用单车位置。
4. 启发用欧氏、图却是四连通：仍可采纳，但偏松。
5. 不计 cap，3 车会把内核跑死。

## 条目小结

| 项目 | 内容 |
|------|------|
| 状态 | 位置元组 |
| 保证 | 单位代价有限图上最优完备 |
| 代价 | \(5^k\) 分支 |
| 下一课 | 放弃联合状态，改成「谁先规划」 |

## 练习

1. 两格走廊对换，手写所有后继，确认全被冲突删掉。
2. 把等待从 opts 拿掉，3×2 例子还能否解？
3. 为何 h=各车曼哈顿之和可采纳？提示：真实 SOC ≥ 无视其它车的最短路之和。


In [ ]:
from lib.studio.widget import PlannerStudio
studio = PlannerStudio()
studio.show_multi(g2.to_studio(), r, "joint A*")
studio
